
# Trains Linear Regression, Random Forest, XGBoost, Mean Predictor,
# Power Persistence across all 10 splits

In [1]:
import gc, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor
from scipy import stats
warnings.filterwarnings("ignore")

In [2]:
BASE_DIR   = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")
FOLDER_VER = "v3_voltage_random10_issue_flight-removed"
FILE_VER   = "v4_voltage_random10_cleaned"
DATA_DIR   = BASE_DIR / f"model_ready_no_dr_{FOLDER_VER}"
OUTPUT_DIR = BASE_DIR / "model_outputs" / "08_baseline_10split"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "target_energy_next_60s_wh"

SPLITS = [
    ("split_01_seed_11",  11),
    ("split_02_seed_22",  22),
    ("split_03_seed_33",  33),
    ("split_04_seed_44",  44),
    ("split_05_seed_55",  55),
    ("split_06_seed_66",  66),
    ("split_07_seed_77",  77),
    ("split_08_seed_88",  88),
    ("split_09_seed_99",  99),
    ("split_10_seed_111", 111),
]

In [3]:
BASE_FEATURES = [
    "battery_status__cell_count", "battery_status__current_a",
    "battery_status__current_filtered_a", "battery_status__discharged_mah",
    "battery_status__remaining", "battery_status__scale",
    "battery_status__voltage_filtered_v", "battery_status__voltage_v",
    "cumulative_energy_wh", "power_w", "time_s",
    "voltage_per_cell", "voltage_filtered_per_cell",
]
def get_models(seed):
    return {
        "Linear Regression": LinearRegression(n_jobs=-1),
        "Random Forest":     RandomForestRegressor(n_estimators=200, max_depth=None,
                                                   random_state=seed, n_jobs=-1),
        "XGBoost":           XGBRegressor(n_estimators=300, learning_rate=0.05,
                                          max_depth=6, subsample=0.8,
                                          colsample_bytree=0.8, random_state=seed,
                                          n_jobs=-1, verbosity=0,
                                          tree_method="hist"),
    }
rows = []

for split_name, seed in SPLITS:
    print(f"\n=== {split_name} ===")
    random.seed(seed); np.random.seed(seed)

    split_dir = DATA_DIR / split_name
    train_df  = pd.read_parquet(split_dir / f"train_no_dr_scaled_{FILE_VER}_{split_name}.parquet")
    test_df   = pd.read_parquet(split_dir / f"test_no_dr_scaled_{FILE_VER}_{split_name}.parquet")

    available = [f for f in BASE_FEATURES if f in train_df.columns]
    X_train = train_df[available].values
    y_train = train_df[TARGET].values
    X_test  = test_df[available].values
    y_test  = test_df[TARGET].values

    # ── Mean Predictor ────────────────────────────────
    train_mean = y_train.mean()
    y_pred_mean = np.full(len(y_test), train_mean)
    rows.append({
        "split": split_name,
        "model": "Mean Predictor",
        "MAE_Wh":  mean_absolute_error(y_test, y_pred_mean),
        "RMSE_Wh": np.sqrt(mean_squared_error(y_test, y_pred_mean)),
        "R2":      r2_score(y_test, y_pred_mean),
    })
    print(f"  Mean Predictor done")

    # ── Power Persistence ─────────────────────────────
    if "power_w" in test_df.columns:
        y_pred_pp = np.maximum(test_df["power_w"].values * (60 / 3600), 0)
        rows.append({
            "split": split_name,
            "model": "Power Persistence",
            "MAE_Wh":  mean_absolute_error(y_test, y_pred_pp),
            "RMSE_Wh": np.sqrt(mean_squared_error(y_test, y_pred_pp)),
            "R2":      r2_score(y_test, y_pred_pp),
        })
        print(f"  Power Persistence done")
    else:
        print("  WARN: power_w not in test_df — skipping Power Persistence")

    # ── LR / RF / XGBoost ────────────────────────────
    for model_name, mdl in get_models(seed).items():
        mdl.fit(X_train, y_train)
        y_pred = mdl.predict(X_test).clip(min=0)
        rows.append({
            "split": split_name,
            "model": model_name,
            "MAE_Wh":  mean_absolute_error(y_test, y_pred),
            "RMSE_Wh": np.sqrt(mean_squared_error(y_test, y_pred)),
            "R2":      r2_score(y_test, y_pred),
        })
        print(f"  {model_name}  MAE={mean_absolute_error(y_test, y_pred):.4f}")
        del mdl; gc.collect()

    del train_df, test_df; gc.collect()

raw = pd.DataFrame(rows).round(4)
raw.to_csv(OUTPUT_DIR / "08_baseline_10split_raw.csv", index=False)
print(f"\nDone: {len(raw)} rows")

# ── Summary table: mean ± std across 10 splits ───────────────
summary = (raw
    .groupby("model")[["MAE_Wh", "RMSE_Wh", "R2"]]
    .agg(["mean", "std"])
    .round(4)
)
summary.columns = ["MAE_mean", "MAE_std", "RMSE_mean", "RMSE_std", "R2_mean", "R2_std"]
summary = summary.sort_values("MAE_mean").reset_index()
summary.index += 1
summary.index.name = "Rank"

print("\n" + "="*80)
print("BASELINE MODELS — 10-SPLIT MEAN ± STD")
print("="*80)
print(summary.to_string())
summary.to_csv(OUTPUT_DIR / "08_baseline_10split_summary.csv")


=== split_01_seed_11 ===
  Mean Predictor done
  Power Persistence done
  Linear Regression  MAE=0.5445
  Random Forest  MAE=1.0261
  XGBoost  MAE=0.9844

=== split_02_seed_22 ===
  Mean Predictor done
  Power Persistence done
  Linear Regression  MAE=0.4322
  Random Forest  MAE=0.6990
  XGBoost  MAE=0.6321

=== split_03_seed_33 ===
  Mean Predictor done
  Power Persistence done
  Linear Regression  MAE=0.2139
  Random Forest  MAE=0.2473
  XGBoost  MAE=0.2652

=== split_04_seed_44 ===
  Mean Predictor done
  Power Persistence done
  Linear Regression  MAE=0.3505
  Random Forest  MAE=0.4418
  XGBoost  MAE=0.3839

=== split_05_seed_55 ===
  Mean Predictor done
  Power Persistence done
  Linear Regression  MAE=0.6342
  Random Forest  MAE=1.3445
  XGBoost  MAE=1.1118

=== split_06_seed_66 ===
  Mean Predictor done
  Power Persistence done
  Linear Regression  MAE=0.6238
  Random Forest  MAE=2.0169
  XGBoost  MAE=1.9353

=== split_07_seed_77 ===
  Mean Predictor done
  Power Persistence do

In [6]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

print("=== 04 baseline summary ===")
print(pd.read_csv(BASE_DIR / "model_outputs/04_baseline_models/baseline_10split_summary.csv").to_string())

print("\n=== 08 baseline summary ===")
print(pd.read_csv(BASE_DIR / "model_outputs/08_baseline_10split/08_baseline_10split_summary.csv").to_string())

print("\n=== 07 raw results ===")
df = pd.read_csv(BASE_DIR / "model_outputs/07_10split_energy/10split_raw_results.csv")
print(df.columns.tolist())
print(df["model"].unique())
print(df["feature_set"].unique())
print(df.to_string())

=== 04 baseline summary ===
               model  test_MAE_mean  test_MAE_std  test_RMSE_mean  test_R2_mean  MAE_normal_mean  MAE_abnormal_mean
0  Linear Regression         0.4509        0.1334          0.9531        0.8498           0.2750             0.7785
1            XGBoost         0.7610        0.4955          1.7455        0.6341           0.3291             1.5411
2      Random Forest         0.7858        0.5247          1.8113        0.5383           0.3767             1.5257

=== 08 baseline summary ===
   Rank              model  MAE_mean  MAE_std  RMSE_mean  RMSE_std  R2_mean  R2_std
0     1  Linear Regression    0.4487   0.1375     0.9536    0.4437   0.8500  0.1425
1     2            XGBoost    0.7914   0.4884     1.7780    1.4292   0.6288  0.2463
2     3      Random Forest    0.8864   0.5372     2.0107    1.4680   0.4062  0.5641
3     4     Mean Predictor    2.1669   0.6209     3.0845    1.6242  -0.1836  0.3120
4     5  Power Persistence    2.3373   0.7967     3.7585   

In [7]:
import pandas as pd
import numpy as np

# ── 1. Baseline models (08 summary — correct 10-split version) ─
baseline = pd.DataFrame([
    {"Model": "Linear Regression", "MAE_mean": 0.4487, "MAE_std": 0.1375,
     "RMSE_mean": 0.9536, "RMSE_std": 0.4437, "R2_mean": 0.8500, "R2_std": 0.1425},
    {"Model": "XGBoost",           "MAE_mean": 0.7914, "MAE_std": 0.4884,
     "RMSE_mean": 1.7780, "RMSE_std": 1.4292, "R2_mean": 0.6288, "R2_std": 0.2463},
    {"Model": "Random Forest",     "MAE_mean": 0.8864, "MAE_std": 0.5372,
     "RMSE_mean": 2.0107, "RMSE_std": 1.4680, "R2_mean": 0.4062, "R2_std": 0.5641},
    {"Model": "Mean Predictor",    "MAE_mean": 2.1669, "MAE_std": 0.6209,
     "RMSE_mean": 3.0845, "RMSE_std": 1.6242, "R2_mean": -0.1836, "R2_std": 0.3120},
    {"Model": "Power Persistence", "MAE_mean": 2.3373, "MAE_std": 0.7967,
     "RMSE_mean": 3.7585, "RMSE_std": 1.7361, "R2_mean": -0.7844, "R2_std": 0.3042},
])

# ── 2. LSTM / BiLSTM from 07 raw — compute overall MAE (normal+abnormal combined) ─
raw = pd.DataFrame({
    "split":       ["split_01_seed_11"]*8 + ["split_02_seed_22"]*8 + ["split_03_seed_33"]*8 +
                   ["split_04_seed_44"]*8 + ["split_05_seed_55"]*8 + ["split_06_seed_66"]*8 +
                   ["split_07_seed_77"]*8 + ["split_08_seed_88"]*8 + ["split_09_seed_99"]*8 +
                   ["split_10_seed_111"]*8,
    "model":       ["LSTM","LSTM","BiLSTM","BiLSTM","LSTM","LSTM","BiLSTM","BiLSTM"]*10,
    "feature_set": ["Base_13","Base_13+CNN_Score","Base_13","Base_13+CNN_Score"]*20,
    "flight_type": ["normal","abnormal"]*40,
    "MAE_Wh": [
        0.2306,1.0379,0.2665,1.0672,0.3053,1.3778,0.2942,1.0536,
        0.6520,0.5053,0.4990,0.5166,0.6455,0.4738,0.5233,0.5443,
        0.1535,0.2979,0.1577,0.2857,0.1357,0.2638,0.1536,0.2631,
        0.6315,0.3481,0.4212,0.3278,0.3587,0.3067,0.6053,0.3509,
        0.5520,0.9908,0.5607,1.2593,0.8982,2.1734,0.5298,0.8528,
        0.1575,4.8374,0.1495,4.2019,0.1519,6.4327,0.1602,6.1075,
        0.1947,1.1374,0.1974,0.8902,0.1955,0.7605,0.1904,1.2265,
        0.1790,1.2075,0.1745,1.2134,0.1847,1.0685,0.1691,0.8239,
        0.2815,0.8171,0.2651,0.7982,0.3047,0.9477,0.2950,0.9022,
        0.1961,1.7704,0.2214,1.7834,0.1822,1.6749,0.2396,1.0026,
    ],
    "RMSE_Wh": [
        0.4298,1.5365,0.5946,1.6638,0.7214,1.9265,0.6174,1.5464,
        1.4571,0.8452,1.1454,0.9102,1.4009,0.7753,1.2090,0.9634,
        0.2589,0.5344,0.2649,0.5587,0.2477,0.5058,0.2566,0.4522,
        1.2088,0.6187,0.7552,0.6180,0.6289,0.5581,1.0495,0.6931,
        1.0909,1.5273,1.0207,2.7609,1.6928,3.5998,0.9816,1.3063,
        0.3140,8.7269,0.3128,7.9122,0.3119,11.4507,0.3294,10.8444,
        0.3899,1.6654,0.3961,1.3478,0.3866,1.1550,0.3786,1.8856,
        0.2823,1.6525,0.2844,1.6201,0.3221,1.5553,0.2921,1.2593,
        0.4682,1.3903,0.4580,1.3174,0.5246,1.5538,0.5056,1.3860,
        0.3424,2.6327,0.3703,2.7609,0.3248,2.4673,0.3827,1.5559,
    ],
})

# Overall MAE per split per model per feature_set (mean of normal+abnormal)
seq_summary = (raw
    .groupby(["split", "model", "feature_set"])[["MAE_Wh","RMSE_Wh"]]
    .mean()
    .reset_index()
)

seq_agg = (seq_summary
    .groupby(["model","feature_set"])[["MAE_Wh","RMSE_Wh"]]
    .agg(["mean","std"])
    .reset_index()
)
seq_agg.columns = ["model","feature_set","MAE_mean","MAE_std","RMSE_mean","RMSE_std"]

# Map to clean model names
def label(row):
    suffix = "" if row["feature_set"] == "Base_13" else " + CNN Score"
    return row["model"] + suffix

seq_agg["Model"] = seq_agg.apply(label, axis=1)
seq_agg["R2_mean"] = np.nan   # not stored in 07 raw
seq_agg["R2_std"]  = np.nan

# ── 3. Combine ────────────────────────────────────────────────
combined = pd.concat([
    baseline[["Model","MAE_mean","MAE_std","RMSE_mean","RMSE_std","R2_mean","R2_std"]],
    seq_agg[["Model","MAE_mean","MAE_std","RMSE_mean","RMSE_std","R2_mean","R2_std"]],
], ignore_index=True)

combined = combined.sort_values("MAE_mean").reset_index(drop=True)
combined.index += 1
combined.index.name = "Rank"

# ── 4. Format: mean ± std ─────────────────────────────────────
def fmt(mean, std):
    if pd.isna(mean): return "-"
    if pd.isna(std):  return f"{mean:.4f}"
    return f"{mean:.4f} ± {std:.4f}"

combined["MAE (Wh)"]  = combined.apply(lambda r: fmt(r["MAE_mean"],  r["MAE_std"]),  axis=1)
combined["RMSE (Wh)"] = combined.apply(lambda r: fmt(r["RMSE_mean"], r["RMSE_std"]), axis=1)
combined["R²"]        = combined.apply(lambda r: fmt(r["R2_mean"],   r["R2_std"]),   axis=1)

final = combined[["Model","MAE (Wh)","RMSE (Wh)","R²"]]

print("="*85)
print("FULL MODEL COMPARISON — 10-Split Mean ± Std")
print("="*85)
print(final.to_string())

# Save
final.to_csv("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/09_full_comparison_table.csv")
print("\nSaved.")

FULL MODEL COMPARISON — 10-Split Mean ± Std
                   Model         MAE (Wh)        RMSE (Wh)                R²
Rank                                                                        
1                 BiLSTM  0.3037 ± 0.1582  0.5802 ± 0.3301                 -
2                   LSTM  0.3295 ± 0.2148  0.6402 ± 0.4495                 -
3      Linear Regression  0.4487 ± 0.1375  0.9536 ± 0.4437   0.8500 ± 0.1425
4                XGBoost  0.7914 ± 0.4884  1.7780 ± 1.4292   0.6288 ± 0.2463
5          Random Forest  0.8864 ± 0.5372  2.0107 ± 1.4680   0.4062 ± 0.5641
6     BiLSTM + CNN Score  1.2736 ± 1.4097  2.1681 ± 2.5904                 -
7       LSTM + CNN Score  1.4215 ± 1.5591  2.3339 ± 2.8164                 -
8         Mean Predictor  2.1669 ± 0.6209  3.0845 ± 1.6242  -0.1836 ± 0.3120
9      Power Persistence  2.3373 ± 0.7967  3.7585 ± 1.7361  -0.7844 ± 0.3042

Saved.


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

# 07 raw results
df = pd.read_csv(BASE_DIR / "model_outputs/07_10split_energy/10split_raw_results.csv")

# Abnormal only
abn = df[df["flight_type"] == "abnormal"].copy()

# 10-split summary
print("=== 10-SPLIT ABNORMAL MAE ===")
s10 = (abn.groupby(["model","feature_set"])["MAE_Wh"]
       .agg(["mean","std"]).round(4))
print(s10.to_string())

# 9-split (exclude split_06)
abn9 = abn[abn["split"] != "split_06_seed_66"]
print("\n=== 9-SPLIT ABNORMAL MAE (excl split_06) ===")
s9 = (abn9.groupby(["model","feature_set"])["MAE_Wh"]
      .agg(["mean","std"]).round(4))
print(s9.to_string())

# Baseline summary
print("\n=== BASELINE 10-SPLIT ===")
print(pd.read_csv(BASE_DIR / "model_outputs/04_baseline_models/baseline_10split_summary.csv").to_string())

=== 10-SPLIT ABNORMAL MAE ===
                            mean     std
model  feature_set                      
BiLSTM Base_13            1.2344  1.1393
       Base_13+CNN_Score  1.3127  1.7126
LSTM   Base_13            1.2950  1.3211
       Base_13+CNN_Score  1.5480  1.8212

=== 9-SPLIT ABNORMAL MAE (excl split_06) ===
                            mean     std
model  feature_set                      
BiLSTM Base_13            0.9046  0.4870
       Base_13+CNN_Score  0.7800  0.3265
LSTM   Base_13            0.9014  0.4698
       Base_13+CNN_Score  1.0052  0.6461

=== BASELINE 10-SPLIT ===
               model  test_MAE_mean  test_MAE_std  test_RMSE_mean  test_R2_mean  MAE_normal_mean  MAE_abnormal_mean
0  Linear Regression         0.4509        0.1334          0.9531        0.8498           0.2750             0.7785
1            XGBoost         0.7610        0.4955          1.7455        0.6341           0.3291             1.5411
2      Random Forest         0.7858        0.5247        

In [8]:
print(seq_summary[seq_summary["split"]=="split_06_seed_66"].to_string())

               split   model        feature_set   MAE_Wh   RMSE_Wh
20  split_06_seed_66  BiLSTM            Base_13  0.15485   0.32110
21  split_06_seed_66  BiLSTM  Base_13+CNN_Score  5.15470   9.37830
22  split_06_seed_66    LSTM            Base_13  0.15470   0.31295
23  split_06_seed_66    LSTM  Base_13+CNN_Score  5.63505  10.08880
